## Assessment 1

Name: Iliana Peters

Student Number: 35723483


### Dataset introduction

The dataset used in this assignment is the Uber Data Analytics Dashboard for the entire 2024 year. It includes the data of various Uber rides, from users and drivers, including dates, locations, timing data, prices, and ratings. A main business case that can be made from this data relates to influential factors in patient satisfaction, leading to the question 'What are the most influential factors that impact rider satisfaction?' Various metrics could be investigated such as as ride distance to booking value, pickup duration, vehicle type, and travel duration to ride distance. Additionally, Customer Cancellation Reasons is a free text column that can be textually analysed for trends and additional insights. 

### Part A: Analytical Query Design and Implementation
*Part A assesses your ability to design, implement, and evaluate a non-trivial analytical query using Apache Spark. The focus is not only on producing correct results, but also on demonstrating an undersatnding of how Spark processes distributed workloads.*


#### 1a. The Business Query
Design and implement a non-trivial business query that requires the following operations:
- A window function, for example, running total, rank within a partition, moving average, cumulative statistics
- High-velocity activity spikes: identify users whose transactions count in any single hour exceeds three standard deviations above their personal hourly mean
- A time-based analysis using date or timestamp attributes


Using the above operations and relating them to the Uber dataset, the following queries are created and will be investigated in the following sections:
1. Calculate statistics on Ride Distance, Booking Value, Pickup Duration, Travel Duration and Driver Rating. These should be ranked by Driver Ratings within the partition. 
2. High-velocity activity spikes: identifyig users who has multiple Booking IDs within a single hour that exceeds three standard deviations above the standard user hourly mean. Correlate these to Drive and Customer Ratings.


#### 1b. Dataset Import and Investigation

In [37]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

#setup a spark session and load dataset
master = "local[*]"
app_name = "Uber Business Query DF"
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('ERROR')

df = spark.read.csv("ncr_ride_bookings.csv",header=True)
df.show(5)
df.printSchema()
df.count()

+----------+--------+----------------+---------------+----------------+-------------+-------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+
|      Date|    Time|      Booking ID| Booking Status|     Customer ID| Vehicle Type|    Pickup Location|    Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|
+----------+--------+----------------+---------------+----------------+-------------+-------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+

150000

In [24]:
#convert numerical columns to float from string
from pyspark.sql.functions import col,concat_ws
from pyspark.sql.types import DateType

# Replace string 'null' with true None/null globally
df = df.replace("null", None)

float_col = ["Avg VTAT","Avg CTAT","Ride Distance","Driver Ratings","Customer Ratings","Booking Value"]
df = df.select([col(c).cast("float") if c in float_col else col(c) for c in df.columns])
df = df.withColumn("Datetime",concat_ws(" ", col("Date"), col("Time")).cast("timestamp"))
df.printSchema()
df.show(5)



root
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Booking ID: string (nullable = true)
 |-- Booking Status: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Vehicle Type: string (nullable = true)
 |-- Pickup Location: string (nullable = true)
 |-- Drop Location: string (nullable = true)
 |-- Avg VTAT: float (nullable = true)
 |-- Avg CTAT: float (nullable = true)
 |-- Cancelled Rides by Customer: string (nullable = true)
 |-- Reason for cancelling by Customer: string (nullable = true)
 |-- Cancelled Rides by Driver: string (nullable = true)
 |-- Driver Cancellation Reason: string (nullable = true)
 |-- Incomplete Rides: string (nullable = true)
 |-- Incomplete Rides Reason: string (nullable = true)
 |-- Booking Value: float (nullable = true)
 |-- Ride Distance: float (nullable = true)
 |-- Driver Ratings: float (nullable = true)
 |-- Customer Rating: string (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Datet

In [25]:
#after confirming the Datetime column matches, the seperate Date and Time columns are dropped
df = df.drop("Date", "Time")
df.show(5)

+----------------+---------------+----------------+-------------+-------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+-------------------+
|      Booking ID| Booking Status|     Customer ID| Vehicle Type|    Pickup Location|    Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|           Datetime|
+----------------+---------------+----------------+-------------+-------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+---

#### 2. DataFrame Implementation

Implement the query using the Spark DataFrame API

##### 2a. Dataframe Investigation
Below is the full workings to create the queries in using Dataframes. Intermediate steps and working notes will be shown

In [26]:
#BUSINESS QUERY 1 - cumulative statistics on the Ride Distance, Booking Value, Pickup Duration, Travel Duration and Driver Rating.
# These should be ranked by Driver Ratings within the partition. 

print(df.count())
#filter out null ratings and store in cache as intermediate dataframe used for the statistical assessment of query 1
filter_1 = df.dropna(subset=["Driver Ratings"]).cache()
print(filter_1.count())
filter_1.describe("Driver Ratings").show()


150000
93000
+-------+------------------+
|summary|    Driver Ratings|
+-------+------------------+
|  count|             93000|
|   mean| 4.230992466042118|
| stddev|0.4368714755910542|
|    min|               3.0|
|    max|               5.0|
+-------+------------------+



In [27]:
#add a column that would specify the customer rating range into 0-3, 3-4, 4-5
from pyspark.sql.functions import when
filter_1 = filter_1.withColumn("Driver_Rating_Range",when(col("Driver Ratings")<=3.0, "0-3").
                               when(col("Driver Ratings")<=3.5, "3-3.5").when(col("Driver Ratings")<=4.0, "3.5-4").
                               when(col("Driver Ratings")<=4.5, "4-4.5").when(col("Driver Ratings")<=5.0, "4.5-5"))
filter_1.show(20)

+----------------+--------------+----------------+-------------+-------------------+----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+-------------------+-------------------+
|      Booking ID|Booking Status|     Customer ID| Vehicle Type|    Pickup Location|   Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|           Datetime|Driver_Rating_Range|
+----------------+--------------+----------------+-------------+-------------------+----------------+--------+--------+---------------------------+---------------------------------+-------------------------+-------------

In [28]:
from pyspark.sql.functions import avg, count, round

stats = (filter_1.groupBy("Driver_Rating_Range").agg(count("*").alias("Count"),round(avg("Driver Ratings"),2).alias("Avg Driver Rating"),
        round(avg("Ride Distance"),2).alias("Avg Ride Distance"),round(avg("Booking Value"),2).alias("Avg Booking Value"),
        round(avg("Avg VTAT"),2).alias("Avg Pickup Duration"),round(avg("Avg CTAT"),2).alias("Avg Travel Duration")))
stats.show()

#clear memory of intermediate dataframe
filter_1.unpersist()

+-------------------+-----+-----------------+-----------------+-----------------+-------------------+-------------------+
|Driver_Rating_Range|Count|Avg Driver Rating|Avg Ride Distance|Avg Booking Value|Avg Pickup Duration|Avg Travel Duration|
+-------------------+-----+-----------------+-----------------+-----------------+-------------------+-------------------+
|              3-3.5| 6697|             3.28|            26.15|           507.39|               8.55|              30.02|
|              4.5-5|23444|             4.74|            25.94|           507.81|               8.48|              30.05|
|                0-3|  745|              3.0|            25.41|           521.87|               8.66|              30.12|
|              4-4.5|46540|             4.28|            25.99|           507.57|               8.52|              30.02|
|              3.5-4|15574|              3.8|            26.08|           510.23|               8.53|              30.07|
+-------------------+---

DataFrame[Booking ID: string, Booking Status: string, Customer ID: string, Vehicle Type: string, Pickup Location: string, Drop Location: string, Avg VTAT: float, Avg CTAT: float, Cancelled Rides by Customer: string, Reason for cancelling by Customer: string, Cancelled Rides by Driver: string, Driver Cancellation Reason: string, Incomplete Rides: string, Incomplete Rides Reason: string, Booking Value: float, Ride Distance: float, Driver Ratings: float, Customer Rating: string, Payment Method: string, Datetime: timestamp, Driver_Rating_Range: string]

In [29]:
#BUSINESS QUERY 2 - high-velocity activity spikes: identifyig users who has multiple Booking IDs within a single hour that exceeds 
# three standard deviations above the standard user hourly mean. Correlate these to Drive and Customer Ratings

#count the number of unique Customer IDs in dataset
unique_count = df.select("Customer ID").distinct().count()
total_count = df.count()
#number of enteries with repeat users
print(total_count - unique_count)

#review the Customer IDs with the most bookings
from pyspark.sql.functions import desc
repeat_bookings = (df.groupBy("Customer ID").agg(count("Booking ID").alias("Repeat_Bookings")).filter(col("Repeat_Bookings") > 1)
    .orderBy(desc("Repeat_Bookings")))
repeat_bookings.show()

#average and standard deviation number of repeat bookings 
from pyspark.sql.functions import stddev
repbook_stats = repeat_bookings.agg(avg("Repeat_Bookings").alias("Avg_Repeat_Bookings"),stddev("Repeat_Bookings").alias("StdDev_Repeat_Bookings"))
repbook_stats.show()


1212
+----------------+---------------+
|     Customer ID|Repeat_Bookings|
+----------------+---------------+
|"""CID7828101"""|              3|
|"""CID4523979"""|              3|
|"""CID6715450"""|              3|
|"""CID6468528"""|              3|
|"""CID8727691"""|              3|
|"""CID5481002"""|              3|
|"""CID9329900"""|              2|
|"""CID3448023"""|              2|
|"""CID5312396"""|              2|
|"""CID1798550"""|              2|
|"""CID5911647"""|              2|
|"""CID5896172"""|              2|
|"""CID7611095"""|              2|
|"""CID6792844"""|              2|
|"""CID7117070"""|              2|
|"""CID9028135"""|              2|
|"""CID5945148"""|              2|
|"""CID9173522"""|              2|
|"""CID4140572"""|              2|
|"""CID5521293"""|              2|
+----------------+---------------+
only showing top 20 rows
+-------------------+----------------------+
|Avg_Repeat_Bookings|StdDev_Repeat_Bookings|
+-------------------+-------------------

In [30]:
from pyspark.sql.functions import lag
from pyspark.sql.window import Window

#dataframe of users with repeat bookings that is cache
repeat_bookings_detail = df.join(repeat_bookings, on="Customer ID", how="inner").cache()
repeat_bookings_detail.count() 

#investigating repeat booking Datetime to see if there are any that happen within the same day and data relating to their bookings
datetime_window = Window.partitionBy("Customer ID").orderBy("Datetime")

#Calculating the time between bookings in hours
velocity_df = (repeat_bookings_detail.withColumn("Prev Booking", lag("Datetime").over(datetime_window)).withColumn("Prev Booking ID",lag("Booking ID").over(datetime_window))
    .withColumn("Time since last Booking",(col("Datetime").cast("long") - col("Prev Booking").cast("long"))/3600))
velocity_df.show()

+----------------+----------------+-------------------+-------------+--------------------+---------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+-------------------+---------------+-------------------+----------------+-----------------------+
|     Customer ID|      Booking ID|     Booking Status| Vehicle Type|     Pickup Location|  Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|           Datetime|Repeat_Bookings|       Prev Booking| Prev Booking ID|Time since last Booking|
+----------------+----------------+-------------------+-------------+--------------------+------

In [31]:
#investigate the process of join in Spark
repeat_bookings_detail.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [Customer ID#17029, Booking ID#17027, Booking Status#17028, Vehicle Type#17030, Pickup Location#17031, Drop Location#17032, Avg VTAT#17046, Avg CTAT#17047, Cancelled Rides by Customer#17035, Reason for cancelling by Customer#17036, Cancelled Rides by Driver#17037, Driver Cancellation Reason#17038, Incomplete Rides#17039, Incomplete Rides Reason#17040, Booking Value#17048, Ride Distance#17049, Driver Ratings#17050, Customer Rating#17044, Payment Method#17045, Datetime#17051, Repeat_Bookings#19509L]
      +- InMemoryRelation [Customer ID#17029, Booking ID#17027, Booking Status#17028, Vehicle Type#17030, Pickup Location#17031, Drop Location#17032, Avg VTAT#17046, Avg CTAT#17047, Cancelled Rides by Customer#17035, Reason for cancelling by Customer#17036, Cancelled Rides by Driver#17037, Driver Cancellation Reason#17038, Incomplete Rides#17039, Incomplete Rides Reason#17040, Booking Value#17048, Ride Distance#17049

In [32]:
#Only showing bookings that are made within 24 hours
velocity_df.filter(col("Time since last Booking") < 24).show()

+----------------+----------------+--------------------+------------+------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+-------------------+---------------+-------------------+----------------+-----------------------+
|     Customer ID|      Booking ID|      Booking Status|Vehicle Type|   Pickup Location|    Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|           Datetime|Repeat_Bookings|       Prev Booking| Prev Booking ID|Time since last Booking|
+----------------+----------------+--------------------+------------+------------------+--------

In [33]:
velocity_df.filter(col("Time since last Booking") < 24).select("Customer ID", "Reason for cancelling by Customer", "Driver Cancellation Reason",
                                                               "Time since last Booking").show(truncate=False)

+----------------+--------------------------------------------+-----------------------------------+-----------------------+
|Customer ID     |Reason for cancelling by Customer           |Driver Cancellation Reason         |Time since last Booking|
+----------------+--------------------------------------------+-----------------------------------+-----------------------+
|"""CID1714179"""|NULL                                        |More than permitted people in there|10.0525                |
|"""CID3211810"""|NULL                                        |NULL                               |9.503333333333334      |
|"""CID3422015"""|NULL                                        |Customer related issue             |20.590555555555557     |
|"""CID4021971"""|NULL                                        |NULL                               |1.7222222222222223     |
|"""CID4061291"""|NULL                                        |NULL                               |11.040277777777778     |
|"""CID7

In [34]:
#clear memory of intermediate dataframe
repeat_bookings_detail.unpersist()

DataFrame[Customer ID: string, Booking ID: string, Booking Status: string, Vehicle Type: string, Pickup Location: string, Drop Location: string, Avg VTAT: float, Avg CTAT: float, Cancelled Rides by Customer: string, Reason for cancelling by Customer: string, Cancelled Rides by Driver: string, Driver Cancellation Reason: string, Incomplete Rides: string, Incomplete Rides Reason: string, Booking Value: float, Ride Distance: float, Driver Ratings: float, Customer Rating: string, Payment Method: string, Datetime: timestamp, Repeat_Bookings: bigint]

##### 2b. Dataframe Query Finalisation

Below is the final code for the three business queries in Spark Dataframe API

In [35]:
from pyspark.sql.functions import lag, col, when, count, avg, count, round, desc
from pyspark.sql.window import Window

#Query 1
query_1 = df.dropna(subset=["Driver Ratings"]).cache()
query_1.count()
query_1 = query_1.withColumn("Driver_Rating_Range",when(col("Driver Ratings")<=3.0, "0-3").
                               when(col("Driver Ratings")<=3.5, "3-3.5").when(col("Driver Ratings")<=4.0, "3.5-4").
                               when(col("Driver Ratings")<=4.5, "4-4.5").when(col("Driver Ratings")<=5.0, "4.5-5"))
stats = (query_1.groupBy("Driver_Rating_Range").agg(count("*").alias("Count"),round(avg("Driver Ratings"),2).alias("Avg Driver Rating"),
        round(avg("Ride Distance"),2).alias("Avg Ride Distance"),round(avg("Booking Value"),2).alias("Avg Booking Value"),
        round(avg("Avg VTAT"),2).alias("Avg Pickup Duration"),round(avg("Avg CTAT"),2).alias("Avg Travel Duration")))
print("The Uber distance, timing and price statistics seperated per rating given by Customer")
stats.show()
query_1.unpersist()

#Query 2
query_2_repeat = (df.groupBy("Customer ID").agg(count("Booking ID").alias("Repeat_Bookings")).filter(col("Repeat_Bookings") > 1)
    .orderBy(desc("Repeat_Bookings")))
from pyspark.sql.functions import stddev
repbook_stats = query_2_repeat.agg(avg("Repeat_Bookings").alias("Avg_Repeat_Bookings"),stddev("Repeat_Bookings").
                                    alias("StdDev_Repeat_Bookings"))
print("The average and standard devaition of repeat bookings made in 2024")
repbook_stats.show()
repeat_bookings_detail = df.join(query_2_repeat, on="Customer ID", how="inner").cache()
repeat_bookings_detail.count() 
datetime_window = Window.partitionBy("Customer ID").orderBy("Datetime")
velocity_df = (repeat_bookings_detail.withColumn("Prev Booking", lag("Datetime").over(datetime_window)).withColumn("Prev Booking ID",lag("Booking ID").over(datetime_window))
    .withColumn("Time since last Booking",(col("Datetime").cast("long") - col("Prev Booking").cast("long"))/3600))
print("The Customer ID with multiple bookings made within 24 hours, with booking status, reasoning and time between bookings")
velocity_df.filter(col("Time since last Booking") < 24).select("Customer ID", "Booking Status", "Reason for cancelling by Customer", 
                            "Driver Cancellation Reason","Time since last Booking").show(truncate=False)
repeat_bookings_detail.unpersist()


The Uber distance, timing and price statistics seperated per rating given by Customer
+-------------------+-----+-----------------+-----------------+-----------------+-------------------+-------------------+
|Driver_Rating_Range|Count|Avg Driver Rating|Avg Ride Distance|Avg Booking Value|Avg Pickup Duration|Avg Travel Duration|
+-------------------+-----+-----------------+-----------------+-----------------+-------------------+-------------------+
|              3-3.5| 6697|             3.28|            26.15|           507.39|               8.55|              30.02|
|              4.5-5|23444|             4.74|            25.94|           507.81|               8.48|              30.05|
|                0-3|  745|              3.0|            25.41|           521.87|               8.66|              30.12|
|              4-4.5|46540|             4.28|            25.99|           507.57|               8.52|              30.02|
|              3.5-4|15574|              3.8|            26.

DataFrame[Customer ID: string, Booking ID: string, Booking Status: string, Vehicle Type: string, Pickup Location: string, Drop Location: string, Avg VTAT: float, Avg CTAT: float, Cancelled Rides by Customer: string, Reason for cancelling by Customer: string, Cancelled Rides by Driver: string, Driver Cancellation Reason: string, Incomplete Rides: string, Incomplete Rides Reason: string, Booking Value: float, Ride Distance: float, Driver Ratings: float, Customer Rating: string, Payment Method: string, Datetime: timestamp, Repeat_Bookings: bigint]

#### 3. Spark SQL Implementation

Implement the same query, or a functionally equivalent version, using Spark SQL

In [36]:
df.createOrReplaceTempView("df_sql")

#Using similar code with SQL to complete query 1
query_1_sql = spark.sql("""SELECT CASE 
        WHEN `Driver Ratings` <= 3.0 THEN '0-3'
        WHEN `Driver Ratings` <= 3.5 THEN '3-3.5'
        WHEN `Driver Ratings` <= 4.0 THEN '3.5-4'
        WHEN `Driver Ratings` <= 4.5 THEN '4-4.5'
        WHEN `Driver Ratings` <= 5.0 THEN '4.5-5'
    END AS Driver_Rating_Range, COUNT(*) AS Count, ROUND(AVG(`Driver Ratings`), 2) AS `Avg Driver Rating`,
    ROUND(AVG(`Ride Distance`), 2) AS `Avg Ride Distance`, ROUND(AVG(`Booking Value`), 2) AS `Avg Booking Value`,
    ROUND(AVG(`Avg VTAT`), 2) AS `Avg Pickup Duration`,ROUND(AVG(`Avg CTAT`), 2) AS `Avg Travel Duration`
    FROM df_sql WHERE `Driver Ratings` IS NOT NULL GROUP BY Driver_Rating_Range """)

print("The Uber distance, timing and price statistics separated per rating given by Customer")
query_1_sql.show()

#Using similar code with SQL to complete query 2

#create temp view for the subset of data that has repeat bookings
spark.sql("""SELECT `Customer ID`, COUNT(`Booking ID`) AS Repeat_Bookings FROM df_sql
    GROUP BY `Customer ID` HAVING COUNT(`Booking ID`) > 1 ORDER BY Repeat_Bookings DESC""").createOrReplaceTempView("query_2_repeat")

#get stats of average and standard deviation for customers with repreat bookings
repbook_stats_sql = spark.sql(""" SELECT AVG(Repeat_Bookings) AS Avg_Repeat_Bookings, STDDEV(Repeat_Bookings) AS StdDev_Repeat_Bookings
    FROM query_2_repeat """)
print("The average and standard deviation of repeat bookings made in 2024")
repbook_stats_sql.show()

#using an CTE to create dataset based on repeat customers and the time between bookings
velocity_sql = spark.sql("""WITH joined AS (SELECT s.*, r.Repeat_Bookings FROM df_sql s
    INNER JOIN query_2_repeat r ON s.`Customer ID` = r.`Customer ID`), 
    windowed AS (SELECT *, LAG(Datetime) OVER (PARTITION BY `Customer ID` ORDER BY Datetime) AS 
    `Prev Booking`, LAG(`Booking ID`) OVER (PARTITION BY `Customer ID` ORDER BY Datetime) AS 
    `Prev Booking ID` FROM joined) SELECT *, (CAST(Datetime AS LONG) - CAST(`Prev Booking` AS LONG)) 
    / 3600 AS `Time since last Booking` FROM windowed""")

velocity_sql.createOrReplaceTempView("velocity")

print("The Customer ID with multiple bookings made within 24 hours, with booking status, reasoning and time between bookings")
spark.sql("""SELECT `Customer ID`, `Booking Status`, `Reason for cancelling by Customer`, `Driver Cancellation Reason`, 
    `Time since last Booking` FROM velocity WHERE `Time since last Booking` < 24""").show(truncate=False)

The Uber distance, timing and price statistics separated per rating given by Customer
+-------------------+-----+-----------------+-----------------+-----------------+-------------------+-------------------+
|Driver_Rating_Range|Count|Avg Driver Rating|Avg Ride Distance|Avg Booking Value|Avg Pickup Duration|Avg Travel Duration|
+-------------------+-----+-----------------+-----------------+-----------------+-------------------+-------------------+
|              3-3.5| 6697|             3.28|            26.15|           507.39|               8.55|              30.02|
|              4.5-5|23444|             4.74|            25.94|           507.81|               8.48|              30.05|
|                0-3|  745|              3.0|            25.41|           521.87|               8.66|              30.12|
|              4-4.5|46540|             4.28|            25.99|           507.57|               8.52|              30.02|
|              3.5-4|15574|              3.8|            26.

#### 4. Result Validaiton

*Demonstrate that both implementations produce equivalent analytical results. Where minor differences occour due to sorting, formatiing, or floating-point precision, provide a brief explaination.*

The two business queries were chosen to provide information relating to customer satisfaction and if there were any common themes relating to ratings given by customers and repeat bookings. They were sufficently complicated due to the additional data operations to group per rating range and timing between bookings per customer. Additionally, mathematical operations were required to calculate averages and time based data for the queries. Various operations and functions were used with both Dataframe API and SQL, however Dataframe API was used to form the structure of the queries and was translated into SQL. 

The output of the queries is impacted by the ordering of the code, as this impacts how much data is utilised during the operation. For example, with operations that only require a singel data column, like withColumn(), and therefore only use that data during the operation, which makes it very efficent. Additionally, Spark projects nessesary columns back through the code and therefore performs the operation on only those columns. In the second query, the final output is only with 5 columns with the information on multiple bookings.  This would be projected back through the previous operations, like join and window function, to optimise the operating power as much as possible. Finally, the join operation was perfromed using BroadcastHashJoin, determined through the explain() function. Since the join was performed on data that was already filtered, only using two columns for this join, this type was used as it is ideal for smaller datasets that can fit in the memory; larger datasets a Sort-Merge join may have been used. 

Even though Dataframe API was used initially, the SQL code is inherently easier to read due to the desciptive nature of the queries. Additionally, the queries can be combined into a single code rather than multiple lines with the Dataframe API. This also increases the computing power of SQL; the time to complete the on my CPU Dataframe API query was 1.9s compared to 1.7s with SQL query. Comparing the output of the two versions, the finalised data is identical including floating point prescision when calculating the average and standard deviation of repeat booking numbers. Additionally the sorting between the two output tables is also identical. 



### Part B: System perspective and performance analysis

*Part B shifts the focus from "does the code work" to "why does it work this way." You will need to demonstrate an understanding of Spark's internal execution model by gathering empical data and producing a written analysis*

#### 1. Partitioning Strategy

Investigate the partitioning strategy with hash and range partitioning using the high cardinality column - Booking ID.

##### 1a. Hash Partitioning

In [72]:
from pyspark.sql.functions import spark_partition_id
from pyspark.sql.functions import min, max, avg, stddev

for partitions in range(1, 7):
    df_hashed = df.repartition(partitions, "Booking ID")
    hash_partition_counts = (df_hashed.withColumn("Booking ID", spark_partition_id()).groupBy("Booking ID").
                            agg(count("*").alias("row_count")))
    hash_skew_metrics = hash_partition_counts.select(min("row_count").alias("min_partition_rows"),max("row_count").alias("max_partition_rows"),
        avg("row_count").alias("avg_partition_rows"),stddev("row_count").alias("stddev_partition_rows"))
    imbalance_ratio = (hash_partition_counts.select((max("row_count") / min("row_count")).alias("imbalance_ratio"))
        .collect()[0]["imbalance_ratio"])

    print(f"Partitions: {partitions} | "f"Imbalance ratio: {imbalance_ratio:.3f}")


Partitions: 1 | Imbalance ratio: 1.000
Partitions: 2 | Imbalance ratio: 1.007
Partitions: 3 | Imbalance ratio: 1.005
Partitions: 4 | Imbalance ratio: 1.015
Partitions: 5 | Imbalance ratio: 1.010
Partitions: 6 | Imbalance ratio: 1.015


In [73]:
#three partitions used as this is still a suitable number to seperate the data and improve computing time while still keeping 
# the dat enteries per partition senesible and meaningful. The dataset is 150,000 entries, so three paritions will not dramatically 
# increase overhead while still able to process the data
partitions = 3

df_hashed = df.repartition(partitions, "Booking ID")
hash_partition_counts = (df_hashed.withColumn("Booking ID", spark_partition_id()).groupBy("Booking ID").
                            agg(count("*").alias("row_count")))
hash_skew_metrics = hash_partition_counts.select(min("row_count").alias("min_partition_rows"),max("row_count").alias("max_partition_rows"),
        avg("row_count").alias("avg_partition_rows"),stddev("row_count").alias("stddev_partition_rows"))

hash_partition_counts.show()
hash_skew_metrics.show()


+----------+---------+
|Booking ID|row_count|
+----------+---------+
|         0|    49854|
|         1|    50122|
|         2|    50024|
+----------+---------+

+------------------+------------------+------------------+---------------------+
|min_partition_rows|max_partition_rows|avg_partition_rows|stddev_partition_rows|
+------------------+------------------+------------------+---------------------+
|             49854|             50122|           50000.0|   135.60235986147143|
+------------------+------------------+------------------+---------------------+



##### 1b. Range Partitioning

In [74]:
df_range = df.repartitionByRange(partitions, "Booking ID")

df_range.withColumn("Booking ID", spark_partition_id()).groupBy("Booking ID").agg(count("*").alias("Row Count")).show()

+----------+---------+
|Booking ID|Row Count|
+----------+---------+
|         0|    51929|
|         1|    45996|
|         2|    52075|
+----------+---------+



In [76]:
range_partition_counts = (df_range.withColumn("Booking ID", spark_partition_id()).groupBy("Booking ID").
                            agg(count("*").alias("row_count")))
range_skew_metrics = range_partition_counts.select(min("row_count").alias("min_partition_rows"),max("row_count").alias("max_partition_rows"),
        avg("row_count").alias("avg_partition_rows"),stddev("row_count").alias("stddev_partition_rows"))
imbalance_ratio = range_partition_counts.select((max("row_count") / min("row_count")).alias("imbalance_ratio"))


range_partition_counts.show()
range_skew_metrics.show()
imbalance_ratio.show()

+----------+---------+
|Booking ID|row_count|
+----------+---------+
|         0|    52925|
|         1|    46365|
|         2|    50710|
+----------+---------+

+------------------+------------------+------------------+---------------------+
|min_partition_rows|max_partition_rows|avg_partition_rows|stddev_partition_rows|
+------------------+------------------+------------------+---------------------+
|             46566|             51921|           50000.0|    2980.919824483711|
+------------------+------------------+------------------+---------------------+

+------------------+
|   imbalance_ratio|
+------------------+
|1.1462901706499595|
+------------------+



##### 1c Comparative Analysis

